- **Runtime :** 14.3 LTS ML (includes Apache Spark 3.5.0, GPU, Scala 2.12) 
- **Driver :** g5.4xlarge [A10G]
- **Librairies**: 
  - poetry==1.5.1

In [0]:
%sh
pip install .

In [0]:
%pip install --force-reinstall --no-deps  neuralforecast-1.7.5-qkcv.tar

dbutils.library.restartPython()

In [0]:
import time, sys, os 
import datetime as dt
print(dt.datetime.now().strftime('%Y%m%d%H%M'))
print('#'*20 + dt.datetime.now().strftime(' %Y.%m.%d ') + '#'*20)

sys.path.append(os.path.abspath("static_feature_in_attention_layer"))
from DB_common_utils import *

dt_current = dt.datetime.now().strftime('%Y%m%d')
dt_current


## <font color=red>load data:  </font>   
<font color=yellow>Meals:  </font> https://www.kaggle.com/datasets/sureshmecad/meal-demand-forecasting/data?select=test.csv  




### <font color=red>Dataset: Meals</font> 

In [0]:
import s3fs,os
fs = s3fs.S3FileSystem()

In [0]:
_use_staging_train = True # skip data proceeding
_save_to_staging = False # update staging file
_no_training = False # skip train
_no_tuning = False # skip tunning
_log_transform = False # if use log

_features_static = ['category', 'cuisine'] + ['city_code', 'region_code', 'center_type', 'op_area']
_features_dynamic = []

In [0]:
### reader
import pyarrow.parquet as pq
import pandas as pd
_p_base = 's3://'

_p_base_folder = os.path.join(_p_base, 'meal-archive')

In [0]:
_df_meals = pd.read_csv(os.path.join(_p_base_folder,'meal_info.csv'))
_df_center = pd.read_csv(os.path.join(_p_base_folder,'fulfilment_center_info.csv'))

In [0]:
import glob

if not _use_staging_train:
    _df_train = pd.read_csv(os.path.join(_p_base_folder, 'train.csv'), sep=',')

    print(f'len {len(_df_train)}')
    _df_train[:10]

    print(f"week.min {_df_train.week.min()}, week.max {_df_train.week.max()}, center_id.max {_df_train.center_id.max()}, num_orders.min {_df_train.num_orders.min()}, num_orders.max() {_df_train.num_orders.max()}")

    unique_id = 'cmid'
    _df_train[unique_id] = _df_train['center_id'] + _df_train['meal_id']*1000

    df_cross_join = _df_train[['week']].drop_duplicates().assign(key=1).merge(_df_train[[unique_id, 'center_id', 'meal_id']].drop_duplicates().assign(key=1), on='key').drop('key', axis=1)

    _df_train['open_flag'] = 1
    _df_train = df_cross_join.merge(_df_train, on=['week', unique_id, 'center_id', 'meal_id'], how='left').sort_values([unique_id, 'week']).reset_index()

    _df_train.count()
    _df_train.isnull().sum()

    _df_train['num_orders'] = _df_train.groupby(unique_id)['num_orders'].ffill()
    _df_train.isnull().sum()

    _df_train['num_orders'].fillna(0.0001, inplace=True)
    _df_train['open_flag'].fillna(0, inplace=True)

    _df_train.isnull().sum()

    _df_train.rename(columns={unique_id: 'unique_id',
                                'num_orders': 'y',
                                }, inplace=True)
    _df_train['ds'] = pd.to_datetime('2020-01-05', format='%Y-%m-%d') + pd.to_timedelta(_df_train['week']*7, unit='d')

    _df_complete_numeric = _df_train.copy()

    _df_train = None
    gc.collect()

    for _c in ['y', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured']:
        _df_complete_numeric[_c].fillna(0.0001, inplace=True)

    if _save_to_staging:

        print(f"saving files, y max {_df_complete_numeric['y'].max()}, min {_df_complete_numeric['y'].min()}")
        _df_complete_numeric.ds.max()

        ### save _df_complete_numeric
        num_splits = 1

        # Split the dataframe into smaller dataframes
        df_splits = np.array_split(_df_complete_numeric, num_splits)

        # Save each split into a separate parquet file
        for i, df_split in enumerate(df_splits):
            df_split.to_parquet(os.path.join(_p_base_folder, f'_df_complete_numeric_part_{i}.parquet'))

else:
    # Read all parquet files in the specified directory
    parquet_files =[]
    for i in range(1):
        parquet_files.append(os.path.join(_p_base_folder, f'_df_complete_numeric_part_{i}.parquet'))
    print(f'parquet_files {parquet_files}')
    # Concatenate all the parquet files into a single dataframe
    _df_complete_numeric = pd.concat([pd.read_parquet(file) for file in parquet_files])

_df_complete_numeric.describe()


In [0]:
### _df_static
_df_complete_numeric.columns

_df_static_numeric = _df_complete_numeric[['unique_id','meal_id','center_id']].drop_duplicates().merge(_df_meals, on='meal_id', how='left').merge(_df_center, on='center_id', how='left')

_df_static_numeric.columns

for _c in _features_static:
    _df_static_numeric[_c] = _df_static_numeric[_c].astype('category').cat.codes

_df_static_numeric

Index(['index', 'week', 'unique_id', 'center_id', 'meal_id', 'id',
       'checkout_price', 'base_price', 'emailer_for_promotion',
       'homepage_featured', 'y', 'open_flag', 'ds'],
      dtype='object')

Index(['unique_id', 'meal_id', 'center_id', 'category', 'cuisine', 'city_code',
       'region_code', 'center_type', 'op_area'],
      dtype='object')

,unique_id,meal_id,center_id,category,cuisine,city_code,region_code,center_type,op_area
0,1062010,1062,10,0,2,17,3,1,27
1,1062011,1062,11,0,2,39,3,0,12
2,1062013,1062,13,0,2,17,3,1,28
3,1062014,1062,14,0,2,34,3,2,4
4,1062017,1062,17,0,2,6,3,0,8
...,...,...,...,...,...,...,...,...,...
3592,2956152,2956,152,4,0,14,1,1,15
3593,2956153,2956,153,4,0,17,3,0,14
3594,2956157,2956,157,4,0,23,7,0,16
3595,2956174,2956,174,4,0,47,3,0,29


## (Optional) <font color=yellow>Training</font> 

In [0]:
gc.collect()
if _no_training:
    raise ValueError(f"Paused before training.")

130

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS,NBEATSx,TFT_v1,TFT
from neuralforecast.losses.pytorch import DistributionLoss,MAE
from neuralforecast.utils import AirPassengersPanel, AirPassengersStatic

if _log_transform:
    _df_complete_numeric['y'] = np.log(_df_complete_numeric['y'])*500

Y_train_df = _df_complete_numeric.loc[(_df_complete_numeric.week >= 20) & (_df_complete_numeric.week < 136), _features_dynamic + ['unique_id', 'ds', 'y']]


In [0]:
_col_forecast='NHITS'

_test_config = {'_n_layers': 3,
 'batch_size': 64,
 'embedding': 256,
 'input_size': 20,
 'learning_rate': 0.001,
 'max_steps': 1000,
 'n_block': 1,
 'n_pool_kernel_size_1': 1,
 'windows_batch_size': 512}

_n_layers = _test_config['_n_layers']
max_steps = _test_config['max_steps']
embedding = _test_config['embedding']
n_block = _test_config['n_block']
n_pool_kernel_size = [_test_config['n_pool_kernel_size_1'], 4, 1] #[1, 4, 1]
n_freq_downsample= [4, 4, 1]

nf = NeuralForecast(
    models=[NHITS(h=10, 
                   
        input_size=_test_config['input_size'],
        exclude_insample_y=False, # the model skips the autoregressive features y[t-input_size:t] if True.<br>
        stack_types= _n_layers * ["identity"], # stacks list in the form N * ['identity'], to be deprecated in favor of `n_stacks`
        n_blocks= _n_layers * [n_block], # Number of blocks for each stack. Note that len(n_blocks) = len(stack_types)
        mlp_units= _n_layers * [[embedding*1, embedding*1]], # Structure of hidden layers for each stack type.
        n_pool_kernel_size=n_pool_kernel_size, # list with the size of the windows to take a max/avg over. 
        n_freq_downsample=n_freq_downsample, # list with the stack's coefficients (inverse expressivity ratios)
        pooling_mode="AvgPool1d", # input pooling module from ['MaxPool1d', 'AvgPool1d']
        interpolation_mode="linear", # interpolation basis from ['linear', 'nearest', 'cubic']
        dropout_prob_theta=0.0, # Float between (0, 1). Dropout for NHITS basis
        activation="ReLU", # activation from ['ReLU', 'Softplus', 'Tanh', 'SELU', 'LeakyReLU', 'PReLU', 'Sigmoid']

        loss=DistributionLoss(distribution='StudentT', level=[10, 50, 90]),
                learning_rate=_test_config['learning_rate'],
                num_lr_decays=-1, # Number of learning rate decays, evenly distributed across max_steps
                early_stop_patience_steps=-1,

                batch_size=_test_config['batch_size'],

                stat_exog_list=_features_static,
                futr_exog_list=_features_dynamic,
                hist_exog_list=[],
                max_steps=max_steps,
                val_check_steps=max_steps,
                scaler_type='robust',
                windows_batch_size=_test_config['windows_batch_size'],
                enable_progress_bar=True,

                ),
    ],
    freq='W'
)
nf.fit(df=Y_train_df, static_df=_df_static_numeric, val_size=30)


In [0]:
Y_test_df = _df_complete_numeric.loc[(_df_complete_numeric.week >= 136) & (_df_complete_numeric.week < 146), _features_dynamic + ['unique_id', 'ds', 'y']]


Y_hat_df = nf.predict(futr_df=Y_test_df
                    #   , df=_df_complete_numeric.loc[(_df_complete_numeric.week >= 1) & (_df_complete_numeric.week < 136), _features_dynamic + ['unique_id', 'ds', 'y']]
                      , static_df=_df_static_numeric)

if _log_transform:
    print("using _log_transform" )
    # Y_hat_df[_col_forecast] = np.exp(Y_hat_df[_col_forecast]).round().astype(int)
    Y_test_df['y'] = np.exp(Y_test_df['y']/500).round().astype(int)

In [0]:
print(f'Y_hat_df {Y_hat_df.columns} {Y_hat_df.ds.min()} {Y_hat_df.ds.max()}')

_df_meged = Y_hat_df.merge(Y_test_df, on=['ds','unique_id'], how='left').fillna(0)

_df_meged['forecast_step'] = _df_meged.sort_values('ds').groupby('unique_id').cumcount() + 1
for _c in [f'{_col_forecast}', f'{_col_forecast}-median', f'{_col_forecast}-lo-90',  f'{_col_forecast}-lo-50', f'{_col_forecast}-hi-50', f'{_col_forecast}-hi-90']:
    if _log_transform:
        _df_meged[_c] = np.exp(_df_meged[_c]/500)
        _df_meged[_c].fillna(0, inplace=True)
        _df_meged[_c].replace([float('inf'), -float('inf')], 4000, inplace=True)
    _df_meged[_c]=_df_meged[_c].round().astype(int)
_df_meged


In [0]:

def wpe_func(forecast_base_ori, eval_horizon=[4, 12, 52], real='y', forecast='TFT-median'):
    objective_metric = 0
    forecast_base = forecast_base_ori.dropna()
    for horizon in eval_horizon:

        wpe = (
            forecast_base[
                (forecast_base.forecast_step <= horizon)
            ]
            .groupby(by=["unique_id"], as_index=False)
            .agg({real: "sum", forecast: "sum"})
        )

        wpe["gap_qty"] = abs(wpe[real] - wpe[forecast])
        wpe["wpe_{}W_qty".format(horizon)] = wpe["gap_qty"] / wpe[real]

        wpe_all = wpe.agg(
            {"gap_qty": "sum", real: "sum",}
        )
        wpe_all["wpe_{}W_qty".format(horizon)] = (
            wpe_all["gap_qty"] / wpe_all[real]
        )

        print(wpe_all["gap_qty"] / wpe_all[real])

    return wpe, wpe_all



In [0]:
#
_op2 = wpe_func(_df_meged, eval_horizon=[30], real='y', forecast=f'{_col_forecast}-median')

print(_op2[0])
print(_op2[1])

0.1989449162032837
      unique_id           y  NHITS-median   gap_qty  wpe_30W_qty
0       1062010   8804.0000          9468  664.0000     0.075420
1       1062011   6993.0000          7425  432.0000     0.061776
2       1062013  10021.0000         10232  211.0000     0.021056
3       1062014   3879.0000          3959   80.0000     0.020624
4       1062017   3758.0000          3003  755.0000     0.200905
...         ...         ...           ...       ...          ...
3592    2956152    664.0002             0  664.0002     1.000000
3593    2956153    856.0000           785   71.0000     0.082944
3594    2956157    926.0000           695  231.0000     0.249460
3595    2956174   1544.0000          1800  256.0000     0.165803
3596    2956186   1220.0000          1220    0.0000     0.000000

[3597 rows x 5 columns]
gap_qty        1.507185e+06
y              7.575891e+06
wpe_30W_qty    1.989449e-01
dtype: float64


In [0]:
def quantile_loss(y_true, y_pred, quantile):
    error = y_true - y_pred
    # print(np.maximum(quantile * error, 0))
    return 2* (
        quantile * np.maximum(error, 0)+ (1 - quantile) * np.maximum(-error, 0)).mean() / np.mean(np.abs(y_true))
    
def calculate_quantile_losses(df_merged, real_col='y'):
    # df_merged = df_forecast.merge(df_real, on=['ds', 'unique_id'], how='left').fillna(0)
    p50_loss = quantile_loss(df_merged[real_col], df_merged[f'{_col_forecast}-median'], 0.5)
    p90_loss_lo = quantile_loss(df_merged[real_col], df_merged[f'{_col_forecast}-lo-90'], 0.9)
    p90_loss_hi = quantile_loss(df_merged[real_col], df_merged[f'{_col_forecast}-hi-90'], 0.9)
    return p50_loss, p90_loss_lo, p90_loss_hi

p50_loss, p90_loss_lo, p90_loss_hi = calculate_quantile_losses(_df_meged)
print(f"P50 Loss: {p50_loss}")
print(f"P90 Loss lo: {p90_loss_lo}")
print(f"P90 Loss hi: {p90_loss_hi}")

print(f"for {_col_forecast}, WPE, P50 Loss, P90 Loss hi\n{_op2[1].iloc[-1]}\n{p50_loss}\n{p90_loss_hi}")

P50 Loss: 0.3343270118502411
P90 Loss lo: 1.8107762110090873
P90 Loss hi: 0.2551681116200017
for NHITS, WPE, P50 Loss, P90 Loss hi
0.1989449162032837
0.3343270118502411
0.2551681116200017


## (Optional) <font color=yellow>Tuning</font> 

In [0]:
if _no_tuning:
    raise ValueError(f"Paused before tuning.")

In [0]:
from hyperopt import hp, fmin, tpe, Trials, space_eval

eval_horizon=[4,10,30]

def calculate_wape_ori(df_forecast, df_actual, tag, log=False):
    df_error = pd.merge(df_forecast, df_actual, how="left").fillna(0)
    df_error = df_error[df_error["week_id"] <= df_actual["week_id"].max()]
    df_error["forecast_step"] = list(range(1, df_error.week_id.nunique() + 1)) * df_error["model_id"].nunique()
    objective_metric = 0
    for h in eval_horizon:
        tmp = df_error[df_error["forecast_step"] <= h]
        wape = np.round(np.sum(np.abs(tmp["sales_quantity"] - tmp["forecast"])) / np.sum(tmp["sales_quantity"]), 3)
        h_min = min(h, tmp["forecast_step"].max())
        if log:
            mlflow.log_metric(f"WAPE_{tag}_{h_min:02d}", wape)
        if h_min in [4, 12, 52]:
            objective_metric += wape
        if h >= df_error["forecast_step"].max():
            break
    return objective_metric

In [0]:
horizon = 10

space = {
    "input_size": hp.choice("input_size", [2*horizon]), #hp.quniform("input_size", horizon, 2*horizon, horizon),
    "_n_layers": hp.choice("_n_layers", [1, 2, 3]),
    "n_block": hp.choice("n_block", [1,2,3]),
    "learning_rate": hp.choice("learning_rate", [0.01,0.001,0.0001]),
    "batch_size": hp.choice("batch_size", [64, 128, 256, 512]),
    "windows_batch_size": hp.choice("windows_batch_size", [256, 512, 1024]),
    "max_steps": hp.choice("max_steps", [1000, 2000, 4000]),
    "embedding": hp.choice("embedding", [64, 128, 256, 512]),
    "n_pool_kernel_size_1": hp.choice("n_pool_kernel_size_1", [1, 4]),

}

In [0]:
random_seed = 42

Y_train_df = _df_complete_numeric.loc[(_df_complete_numeric.week >= 20) & (_df_complete_numeric.week < 126), _features_dynamic + ['unique_id', 'ds', 'y']]
Y_valid_df = _df_complete_numeric.loc[(_df_complete_numeric.week >= 126) & (_df_complete_numeric.week < 136), _features_dynamic + ['unique_id', 'ds', 'y']]

print(f'Y_valid_df {Y_valid_df.ds.min()} {Y_valid_df.ds.max()}')

def calculate_wape_metric(df_error, eval_horizon=[4, 12, 52], tag='', log=False):
    print(f"calc WAPE_{tag}, df_error contains {df_error.columns}")
    objective_metric = 0
    for h in eval_horizon:
        tmp = df_error[df_error["forecast_step"] <= h]
        wape = np.round(np.sum(np.abs(tmp["y"] - tmp["TFT"])) / np.sum(tmp["y"]), 5)
        h_min = min(h, tmp["forecast_step"].max())
        
        print(f"calc WAPE_{tag}_{h_min:02d} wape {wape}")
        if h_min in [4, 12, 52]:
            objective_metric += wape
        if h >= df_error["forecast_step"].max():
            print(f"h = {h}, while max step is {df_error.forecast_step.max()}")
            break
    return objective_metric

def calculate_quantile_losses_metric(df_merged, eval_horizon=[10], real_col='y'):
    p50_loss = quantile_loss(df_merged[real_col], df_merged[f'{_col_forecast}-median'], 0.5)
    # p90_loss = quantile_loss(df_merged[real_col], df_merged[forecast_col], 0.9)
    print(f"calc p50_loss {p50_loss}")
    return p50_loss

def fn(params): 

    input_size = int(params["input_size"])
    _n_layers = int(params["_n_layers"])
    learning_rate = params["learning_rate"]
    max_steps = int(params["max_steps"])
    batch_size = int(params["batch_size"]) #int(2**params["batch_size_exponent"])
    windows_batch_size = int(params["windows_batch_size"]) if params["windows_batch_size"] else None 
    n_pool_kernel_size_1 = int(params["n_pool_kernel_size_1"])

    n_pool_kernel_size = [n_pool_kernel_size_1, 4, 1]
    n_freq_downsample= [4, 4, 1 ]
    n_block = int(params["n_block"])

    embedding = int(params["embedding"])

    tft = NHITS(
        h=horizon,
        input_size=input_size,

        exclude_insample_y=False, # the model skips the autoregressive features y[t-input_size:t] if True.<br>
        stack_types= _n_layers * ["identity"], # stacks list in the form N * ['identity'], to be deprecated in favor of `n_stacks`
        n_blocks= _n_layers * [n_block], # Number of blocks for each stack. Note that len(n_blocks) = len(stack_types)
        mlp_units= _n_layers * [[embedding*1, embedding*1]], # Structure of hidden layers for each stack type.
        n_pool_kernel_size=n_pool_kernel_size, # list with the size of the windows to take a max/avg over. 
        n_freq_downsample=n_freq_downsample, # list with the stack's coefficients (inverse expressivity ratios)
        pooling_mode="AvgPool1d", # input pooling module from ['MaxPool1d', 'AvgPool1d']
        interpolation_mode="linear", # interpolation basis from ['linear', 'nearest', 'cubic']
        dropout_prob_theta=0.0, # Float between (0, 1). Dropout for NHITS basis
        activation="ReLU", # activation from ['ReLU', 'Softplus', 'Tanh', 'SELU', 'LeakyReLU', 'PReLU', 'Sigmoid']

        loss=DistributionLoss(distribution='StudentT', level=[10, 50, 90]),
        learning_rate=learning_rate,
        num_lr_decays=-1, # Number of learning rate decays, evenly distributed across max_steps
        early_stop_patience_steps=-1,

        batch_size=batch_size,

        stat_exog_list=_features_static,
        futr_exog_list=_features_dynamic,
        hist_exog_list=[],
        max_steps=max_steps,
        val_check_steps=max_steps+1,
        scaler_type='robust',
        windows_batch_size=windows_batch_size,
        enable_progress_bar=True,
        start_padding_enabled=True,
    )
        
    forecastor = NeuralForecast(models=[tft], freq='W')
    forecastor.fit(df=Y_train_df, static_df=_df_static_numeric, val_size=30)
    Y_hat_df = forecastor.predict(futr_df=Y_valid_df, static_df=_df_static_numeric)

    _df_meged = Y_hat_df.merge(Y_valid_df, on=['ds','unique_id'], how='left').fillna(0)
    _df_meged['forecast_step'] = _df_meged.sort_values('ds').groupby('unique_id').cumcount() + 1
    for _c in [f'{_col_forecast}', f'{_col_forecast}-median', f'{_col_forecast}-lo-90', f'{_col_forecast}-lo-50', f'{_col_forecast}-hi-50', f'{_col_forecast}-hi-90']:
        _df_meged[_c]=_df_meged[_c].round().astype(int)
    # objective_metric_rec = calculate_wape_metric(_df_meged, eval_horizon=[4, 30])

    return calculate_quantile_losses_metric(_df_meged)

Y_valid_df 2022-06-05 00:00:00 2022-08-07 00:00:00


In [0]:
max_evals = 50
timeout = None #3600 * 2
trials = Trials()

best = fmin(
    fn=fn, 
    space=space, # Ensure no duplicate keys in space
    algo=tpe.suggest, 
    max_evals=max_evals,
    timeout=timeout,
    trials=trials,
    rstate=np.random.default_rng(random_seed), 
    catch_eval_exceptions=True,
    show_progressbar=True,
)

In [0]:
space_eval(space, best)

{'_n_layers': 3,
 'batch_size': 64,
 'embedding': 256,
 'input_size': 20,
 'learning_rate': 0.001,
 'max_steps': 1000,
 'n_block': 1,
 'n_pool_kernel_size_1': 1,
 'windows_batch_size': 512}

In [0]:
res_df = (
    pd.DataFrame({"tid" : trials.tids, "tloss" : trials.losses()})
    .join(pd.DataFrame([space_eval(space, {x[0]: x[1][0] for x in trial['misc']['vals'].items()}) for trial in trials.trials]))
)
res_df.display()